<a href="https://colab.research.google.com/github/mauriciobrito31-ui/BA320HW/blob/main/MBrito%20Q3.2.1%20Lab%20multiple%20Re.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment: Multiple Regression for Flight Delay Prediction (Final Version)
## Objective
The goal is to build a multiple regression model to predict flight arrival delays using two independent variables: departure delay and weather. We will learn how to use the formula feature in `statsmodels` to explicitly set a baseline for a categorical variable ('Weather').
#
### Steps:
1.  **Connect to Database & Engineer Variables:** Write an SQL query to calculate `departure_delay` and retrieve `weather_condition` data.
2.  **Data Cleaning:** Remove null values and extreme outliers to improve model stability.
3.  **Build the Multiple Regression Model:** Create the model using `departure_delay` and `weather_condition` as independent variables, setting 'Clear' as the baseline for weather.
4.  **Analyze the Results:** Check the model's statistical significance and the impact of each variable on arrival delay.
5.  **Visualize the Results:** Use `seaborn` to visually analyze the model's performance and the effect of weather.


In [1]:
!pip install pymysql

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 1.1 MB/s eta 0:00:00


In [2]:
# Step 0: Import necessary libraries
import pymysql
import pandas as pd
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')  # Hide warning messages for cleaner output


## Step 1: Connect to DB and Load Data
We connect to the database and load data from the flights_delay table which contains all the engineered features we need.


In [3]:
# Create a direct connection to the database
conn = pymysql.connect(
    host='database-klee.cbgcswckszgl.us-east-1.rds.amazonaws.com',
    user='erau',
    password='1212',
    database='airline_db',
    charset='utf8mb4'
)

# Query data from the flights_delay table
query = """
SELECT
    flight_id,
    departure_delay_mins,
    delay_minutes AS arrival_delay_mins,
    weather_condition,
    day_of_week,
    aircraft_type,
    airline_code
FROM
    flights_delay
WHERE
    departure_delay_mins IS NOT NULL
    AND delay_minutes IS NOT NULL
    AND weather_condition IS NOT NULL;
"""

# Execute the query and load data into a DataFrame
df = pd.read_sql_query(query, conn)

# Close the database connection
conn.close()

In [4]:
df.head()

,flight_id,departure_delay_mins,arrival_delay_mins,weather_condition,day_of_week,aircraft_type,airline_code
0,1,6,22,Clear,Saturday,A330-300,DL
1,2,20,62,Snow,Sunday,A330-300,OZ
2,3,12,28,Rain,Monday,A350-900,EK
3,4,8,15,Cloudy,Monday,A350-900,NH
4,5,15,27,Clear,Monday,B787-9,OZ


## Step 2: Data Cleaning and Preparation
We clean the data by removing any missing values and filtering out extreme outliers, which could negatively impact the model's predictive power.


In [5]:
# Check for missing values before cleaning
print("Missing values per column:")
print(df.isnull().sum())

# Only drop rows with missing values in critical columns
df_cleaned = df.dropna(subset=['departure_delay_mins', 'arrival_delay_mins', 'weather_condition'])

# Use cleaned dataset for further analysis
df_cleaned

Missing values per column:
flight_id               0
departure_delay_mins    0
arrival_delay_mins      0
weather_condition       0
day_of_week             0
aircraft_type           0
airline_code            0
dtype: int64


,flight_id,departure_delay_mins,arrival_delay_mins,weather_condition,day_of_week,aircraft_type,airline_code
0,1,6,22,Clear,Saturday,A330-300,DL
1,2,20,62,Snow,Sunday,A330-300,OZ
2,3,12,28,Rain,Monday,A350-900,EK
3,4,8,15,Cloudy,Monday,A350-900,NH
4,5,15,27,Clear,Monday,B787-9,OZ
...,...,...,...,...,...,...,...
495,496,4,1,Clear,Thursday,B787-9,LH
496,497,0,0,Cloudy,Friday,B787-9,SQ
497,498,9,15,Clear,Tuesday,A380-800,OZ
498,499,29,60,Thunderstorm,Wednesday,B737-MAX,JL


## Step 3: Build the Multiple Regression Model
Build the multiple regression model using `smf.ols`.

dv: arrival_delay_mins ivs: departure_delay_mins,weather_condition

Inside the formula, Use the syntax `C(weather_condition, Treatment(reference='Clear'))` to explicitly set 'Clear' as the baseline category for weather.


In [9]:
# Q1 - Build the model using 'Clear' as the reference category (note the capital C)
smf.ols('arrival_delay_mins ~ departure_delay_mins + C(weather_condition, Treatment(reference="Clear"))', data=df_cleaned).fit().summary()


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:     arrival_delay_mins   R-squared:                       0.890
Model:                            OLS   Adj. R-squared:                  0.889
Method:                 Least Squares   F-statistic:                     668.0
Date:                Sat, 04 Oct 2025   Prob (F-statistic):          4.26e-233
Time:                        21:41:10   Log-Likelihood:                -1787.4
No. Observations:                 500   AIC:                             3589.
Df Residuals:                     493   BIC:                             3618.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
======================================================================================================================================
                                                                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------------------------------------
Intercept                                                              4.8010      0.669      7.172      0.000       3.486       6.116
C(weather_condition, Treatment(reference="Clear"))[T.Cloudy]           3.4593      0.978      3.537      0.000       1.537       5.381
C(weather_condition, Treatment(reference="Clear"))[T.Fog]             19.3783      1.480     13.095      0.000      16.471      22.286
C(weather_condition, Treatment(reference="Clear"))[T.Rain]             8.4370      1.187      7.106      0.000       6.104      10.770
C(weather_condition, Treatment(reference="Clear"))[T.Snow]            30.3141      1.757     17.257      0.000      26.863      33.766
C(weather_condition, Treatment(reference="Clear"))[T.Thunderstorm]    31.5519      2.625     12.019      0.000      26.394      36.710
departure_delay_mins                                                   1.1638      0.023     51.162      0.000       1.119       1.209
==============================================================================
Omnibus:                        2.023   Durbin-Watson:                   2.058
Prob(Omnibus):                  0.364   Jarque-Bera (JB):                2.084
Skew:                           0.127   Prob(JB):                        0.353
Kurtosis:                       2.812   Cond. No.                         172.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

## Step 4: Review and Interpret the Results
The model summary will show the coefficient for `departure_delay_mins` as well as the additional impact of other weather conditions compared to 'Clear' weather.


In [10]:
# Q2 - Print the regression results summary
results = smf.ols('arrival_delay_mins ~ departure_delay_mins + C(weather_condition, Treatment(reference="Clear"))', data=df_cleaned).fit()
print(results.summary())



                            OLS Regression Results                            
Dep. Variable:     arrival_delay_mins   R-squared:                       0.890
Model:                            OLS   Adj. R-squared:                  0.889
Method:                 Least Squares   F-statistic:                     668.0
Date:                Sat, 04 Oct 2025   Prob (F-statistic):          4.26e-233
Time:                        21:59:31   Log-Likelihood:                -1787.4
No. Observations:                 500   AIC:                             3589.
Df Residuals:                     493   BIC:                             3618.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                                                         coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------

## Step 5: Visualize the Results
We analyze the model with two visualizations:
1.  A scatter plot comparing the actual and predicted values to check the overall predictive power of the model.
2.  A bar chart comparing the average arrival delay for each weather condition.


In [8]:
# 1. Visualize the overall model fit (Actual vs. Predicted)
df_cleaned['predicted_delay'] = results.fittedvalues

plt.figure(figsize=(10, 6))
sns.scatterplot(x='predicted_delay', y='arrival_delay_mins', data=df_cleaned)
plt.plot([df_cleaned['arrival_delay_mins'].min(), df_cleaned['arrival_delay_mins'].max()],
         [df_cleaned['arrival_delay_mins'].min(), df_cleaned['arrival_delay_mins'].max()],'r--', lw=2)


NameError: name 'results' is not defined

In [ ]:
# 2. Visualize the effect of weather condition on arrival delay
plt.figure(figsize=(10, 3))
weather_means = df_cleaned.groupby('weather_condition')['arrival_delay_mins'].mean().sort_values()
sns.barplot(x=weather_means.index, y=weather_means.values)
plt.title('Average Arrival Delay by Weather Condition')
plt.xlabel('Weather Condition')
plt.ylabel('Average Arrival Delay (minutes)')


In [ ]:
# Print summary statistics by weather condition
df_cleaned.groupby('weather_condition')['arrival_delay_mins'].describe()